# 04. 공급기업 기술 문서 통합

특허·제품·솔루션 데이터를 기술 문서 단위로 통합하고, 검색에 사용할 문장을 만듭니다. 입력 파일은 비공개 원본을 정리한 중간 파일이며 공개 저장소에는 포함하지 않습니다.

In [ ]:
from pathlib import Path

import pandas as pd

INPUT_PATH = Path('../data/raw/technology_records.csv')
OUTPUT_PATH = Path('../artifacts/technology_embedding_input.csv')
REQUIRED_COLUMNS = {'provider_company', 'technology_type', 'technology_name', 'description'}


def build_technology_embedding_text(technology_df: pd.DataFrame) -> pd.DataFrame:
    missing_columns = REQUIRED_COLUMNS - set(technology_df.columns)
    if missing_columns:
        raise KeyError(f'Missing required columns: {sorted(missing_columns)}')

    result_df = technology_df.copy()
    result_df['embedding_text'] = (
        result_df['technology_type'].fillna('').astype(str).str.strip() + ' ' +
        result_df['technology_name'].fillna('').astype(str).str.strip() + ' ' +
        result_df['description'].fillna('').astype(str).str.strip()
    ).str.replace(r'\s+', ' ', regex=True).str.strip()
    return result_df.loc[result_df['embedding_text'].ne('')]


if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Private input is not available: {INPUT_PATH}')

technology_df = pd.read_csv(INPUT_PATH)
embedding_input_df = build_technology_embedding_text(technology_df)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
embedding_input_df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
embedding_input_df[['provider_company', 'technology_type', 'technology_name', 'embedding_text']].head()